## Power sweep — optimised waveform

Scales the optimised waveform amplitude across a range of peak powers
(shape unchanged) and runs multi-trial simulations at each level.
Produces overlaid PSTHs and a power × time heatmap to show how
the evoked activity profile changes with drive strength.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
from pathlib import Path

plt.style.use(Path('..') / 'configs' / 'mpl.mplstyle')

from designer_waveform.models import RandomEINetwork, load_config
from designer_waveform.optics import OpticsConfig, SigmoidPowerCurve
from designer_waveform.waveforms import RectangularPulseWaveform, PulseTrainWaveform

In [ ]:
# ── Waveform to sweep ─────────────────────────────────────────────────────
OPT_RESULT_PATH = Path('../results/optimised_waveforms/increasing_l23/l23/increasing_l23_l23_optimisation_result.pkl')

# ── Power levels to sweep (mW, peak source power) ────────────────────────
# These scale the waveform amplitude while keeping the shape fixed.
# Set to None to auto-generate N_LEVELS linspace levels up to MAX_POWER_MW.
POWER_LEVELS_MW = None    # e.g. [0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 4.55]
N_LEVELS        = 8       # used only when POWER_LEVELS_MW is None
MIN_POWER_MW    = 0.2     # lower bound of auto-generated range

# ── Fine low-power sweep (separate run, separate plots) ───────────────────
FINE_MIN_POWER_MW   = 0.02   # lower bound (mW)
FINE_MAX_POWER_MW   = 0.5    # upper bound (mW)
FINE_N_LEVELS       = 12     # number of levels

# ── Silence padding ───────────────────────────────────────────────────────
PRE_SILENCE_MS  = 20.0
POST_SILENCE_MS = 100.0

# ── PSTH bin size ─────────────────────────────────────────────────────────
BIN_SIZE_MS = 10.0

# ── Multi-run settings ───────────────────────────────────────────────────
N_RUNS    = 10      # runs per power level (reduce for speed)
SEED_BASE = 1000

VARY_INIT_V       = True
VARY_CONNECTIVITY = True
VARY_WEIGHTS      = True

# ── Network ──────────────────────────────────────────────────────────────
N_EXC     = 8000
N_INH     = 2000
T_PRE_MS  = 200.0
T_POST_MS = 100.0

# ── Energy-matched comparison waveforms ──────────────────────────────────
# At each power level, also simulate:
#   rect : single rectangular pulse, same duration, energy-matched
#   pt   : pulse train, same freq/pulse-width, same duration, energy-matched
CMP_PT_FREQ_HZ      = 25.0   # Hz
CMP_PT_PULSE_DUR_MS = 10.0   # ms per pulse

# ── Opsin ─────────────────────────────────────────────────────────────────
# 'c1v1'    : C1V1-A  (i_max ≈ 1175 pA, K½ ≈ 0.0015 mW/mm²)
# 'chrmine' : ChRmine (i_max ≈ 4600 pA, K½ ≈ 0.037  mW/mm²)
OPSIN = 'c1v1'

In [ ]:
with open(OPT_RESULT_PATH, 'rb') as f:
    _opt = pickle.load(f)

_inner_wf      = _opt['opt_waveform']
_orig_stim_dur = float(_opt['stim_dur_ms'])
_source        = _opt['source']
_layer         = _opt['target_layer']

# Characterise the waveform at native amplitude
_t_eval           = np.linspace(0, _orig_stim_dur, 20_000)
_w_eval           = np.clip(_inner_wf(_t_eval), 0, None)
_peak_native      = float(_w_eval.max())
_energy_native    = float(np.trapz(_w_eval, _t_eval))   # mW·ms
_avg_power_native = _energy_native / _orig_stim_dur     # mW

print(f'Loaded: {_source} / {_layer}')
print(f'  inner duration : {_orig_stim_dur:.1f} ms')
print(f'  native peak    : {_peak_native:.4f} mW')
print(f'  native energy  : {_energy_native:.3f} mW·ms')
print(f'  native avg pwr : {_avg_power_native:.4f} mW')

STIM_DUR_MS = PRE_SILENCE_MS + _orig_stim_dur + POST_SILENCE_MS
print(f'  total window   : {STIM_DUR_MS:.0f} ms  '
      f'({PRE_SILENCE_MS:.0f} pre + {_orig_stim_dur:.0f} active + {POST_SILENCE_MS:.0f} post)')

In [ ]:
CONFIG_PATH = Path('..') / 'configs' / 'random_ei.json'
cfg = load_config(CONFIG_PATH)
cfg.N_exc       = N_EXC
cfg.N_inh       = N_INH
cfg.t_pre_ms    = T_PRE_MS
cfg.t_post_ms   = T_POST_MS
cfg.t_stim_ms   = STIM_DUR_MS
cfg.psth_bin_ms = BIN_SIZE_MS

_optics = OpticsConfig.from_file(Path('..') / 'data' / 'optics_params.json')
_curve  = {'c1v1': SigmoidPowerCurve.c1v1, 'chrmine': SigmoidPowerCurve.chrmine}[OPSIN]()
model   = RandomEINetwork(cfg, optics=_optics, power_curve=_curve,
                          normalization='max_expression')

MAX_POWER_MW = _optics.area_mm2 * 0.1 / _optics.total_transmission
print(f'Model built.  Max deliverable power: {MAX_POWER_MW:.2f} mW')
print(f'Opsin: {OPSIN}  i_max={_curve.i_max_pA:.0f} pA  K½={_curve.half_sat_mW_mm2} mW/mm²')
print(f'Opsin mean: {model._stim_dist_pA.mean():.1f} pA, '
      f'frac zero: {(model._stim_dist_pA == 0).mean():.3f}')

# ── Resolve power levels ──────────────────────────────────────────────────
if POWER_LEVELS_MW is not None:
    power_levels = np.array(POWER_LEVELS_MW, dtype=float)
else:
    power_levels = np.linspace(MIN_POWER_MW, MAX_POWER_MW, N_LEVELS)

print(f'\nPower levels ({len(power_levels)}): '
      + ', '.join(f'{p:.2f}' for p in power_levels) + ' mW')

OUTPUT_DIR = Path('../results') / 'power_sweep' / f'{_source}_{_layer.replace("/","_")}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Output dir: {OUTPUT_DIR}')

In [ ]:
# ── Waveform factory functions ────────────────────────────────────────────
# All three return a padded callable matching STIM_DUR_MS.
# Rect and PT are energy-matched to the optimised waveform at the same power level.

# Energy of the unit-amplitude pulse train (for PT scaling)
_pt_unit = PulseTrainWaveform(
    onset_ms=0.0, pulse_duration_ms=CMP_PT_PULSE_DUR_MS,
    frequency_hz=CMP_PT_FREQ_HZ, train_duration_ms=_orig_stim_dur, amplitude=1.0,
)
_pt_unit_energy = float(np.trapz(np.clip(_pt_unit(_t_eval), 0, None), _t_eval))
print(f'PT unit energy : {_pt_unit_energy:.3f} mW·ms  '
      f'({CMP_PT_FREQ_HZ:.0f} Hz, {CMP_PT_PULSE_DUR_MS:.0f} ms pulses, {_orig_stim_dur:.0f} ms)')

def _make_waveform(p_mw):
    """Optimised waveform scaled to target peak power."""
    _s = p_mw / _peak_native
    def _wf(t, _w=_inner_wf, _p=PRE_SILENCE_MS, _d=_orig_stim_dur, _sc=_s):
        t = np.asarray(t, dtype=float)
        mask = (t >= _p) & (t < _p + _d)
        out  = np.zeros_like(t)
        if mask.any():
            out[mask] = np.clip(_w(t[mask] - _p) * _sc, 0, None)
        return out
    return _wf

def _make_rect(p_mw):
    """Energy-matched rectangle: same duration, avg-power amplitude."""
    _amp  = _avg_power_native * (p_mw / _peak_native)
    _rect = RectangularPulseWaveform(onset_ms=0.0, duration_ms=_orig_stim_dur, amplitude=_amp)
    def _wf(t, _w=_rect, _p=PRE_SILENCE_MS, _d=_orig_stim_dur):
        t = np.asarray(t, dtype=float)
        mask = (t >= _p) & (t < _p + _d)
        out  = np.zeros_like(t)
        if mask.any():
            out[mask] = _w(t[mask] - _p)
        return out
    return _wf

def _make_pt(p_mw):
    """Energy-matched pulse train: same freq/dur, amplitude scaled to match energy."""
    _amp = _energy_native * (p_mw / _peak_native) / _pt_unit_energy
    _pt  = PulseTrainWaveform(
        onset_ms=0.0, pulse_duration_ms=CMP_PT_PULSE_DUR_MS,
        frequency_hz=CMP_PT_FREQ_HZ, train_duration_ms=_orig_stim_dur, amplitude=_amp,
    )
    def _wf(t, _w=_pt, _p=PRE_SILENCE_MS, _d=_orig_stim_dur):
        t = np.asarray(t, dtype=float)
        mask = (t >= _p) & (t < _p + _d)
        out  = np.zeros_like(t)
        if mask.any():
            out[mask] = _w(t[mask] - _p)
        return out
    return _wf

_makers = [('opt', _make_waveform), ('rect', _make_rect), ('pt', _make_pt)]

# ── Run sweep ─────────────────────────────────────────────────────────────
sweep_results = []

for idx, p_mw in enumerate(power_levels):
    print(f'[{idx+1}/{len(power_levels)}]  {p_mw:.3f} mW ...', flush=True)
    _level = {'power_mw': p_mw}

    for cname, maker in _makers:
        wf = maker(p_mw)
        _runs_hz = []
        for _i in range(N_RUNS):
            _r = model.run(wf, seed=SEED_BASE + _i,
                           vary_init_v=VARY_INIT_V,
                           vary_connectivity=VARY_CONNECTIVITY,
                           vary_weights=VARY_WEIGHTS)
            _runs_hz.append(_r['psth_exc'] / (BIN_SIZE_MS / 1000.0))
        _arr = np.stack(_runs_hz)
        _level[cname] = {
            'mean_hz': _arr.mean(0),
            'sem_hz':  _arr.std(0) / np.sqrt(N_RUNS),
            'peak_hz': float(_arr.mean(0).max()),
        }

    sweep_results.append(_level)
    print(f'   peak — opt: {_level["opt"]["peak_hz"]:.1f} Hz  '
          f'rect: {_level["rect"]["peak_hz"]:.1f} Hz  '
          f'pt: {_level["pt"]["peak_hz"]:.1f} Hz')

t_psth_ms = model.run(_make_waveform(power_levels[0]))['t_psth_ms']
print('\nSweep complete.')

In [ ]:
# ── Waveform + PSTH overlay, one column per condition ─────────────────────
# Top row: stimulus waveforms at each power level (plasma, same colour map)
# Bottom row: mean ± SEM PSTHs, shared x-axis with the waveform above

cmap   = plt.cm.plasma
p_norm = plt.Normalize(power_levels.min(), power_levels.max())

_cond_info = [
    ('opt',  'Optimised',                _make_waveform),
    ('rect', 'Energy-matched square pulse', _make_rect),
    ('pt',   f'Energy-matched pulse train\n({CMP_PT_FREQ_HZ:.0f} Hz, {CMP_PT_PULSE_DUR_MS:.0f} ms pulses)', _make_pt),
]

# Fine time grid for waveform evaluation (0.2 ms resolution)
_t_wf = np.linspace(0, STIM_DUR_MS, int(STIM_DUR_MS / 0.2))

fig, axes = plt.subplots(2, 3, figsize=(18, 6),
                          gridspec_kw={'height_ratios': [1, 2.5]},
                          sharex='col')

for col, (ckey, ctitle, maker) in enumerate(_cond_info):
    ax_wf   = axes[0, col]
    ax_psth = axes[1, col]

    # Waveforms (top)
    for p_mw in power_levels:
        color = cmap(p_norm(p_mw))
        ax_wf.plot(_t_wf, maker(p_mw)(_t_wf), color=color, lw=1.2)
    ax_wf.set_xlim(0, STIM_DUR_MS)
    ax_wf.set_ylabel('Power (mW)')
    ax_wf.set_title(ctitle)
    ax_wf.tick_params(bottom=False, labelbottom=False)

    # PSTHs (bottom)
    for res in sweep_results:
        color = cmap(p_norm(res['power_mw']))
        d = res[ckey]
        ax_psth.fill_between(t_psth_ms, d['mean_hz'] - d['sem_hz'], d['mean_hz'] + d['sem_hz'],
                             color=color, alpha=0.15)
        ax_psth.plot(t_psth_ms, d['mean_hz'], color=color, lw=1.5)
    ax_psth.set_xlim(0, STIM_DUR_MS)
    ax_psth.set_xlabel('Time from stim onset (ms)')

axes[1, 0].set_ylabel('Firing rate (Hz)')
fig.suptitle(f'{_source} / {_layer}  ({N_RUNS} runs each)', y=1.01)

# Colorbar in its own reserved strip — tight_layout first, then make room
fig.tight_layout()
fig.subplots_adjust(right=0.87)
cax = fig.add_axes([0.89, 0.1, 0.015, 0.8])
sm = plt.cm.ScalarMappable(cmap=cmap, norm=p_norm)
sm.set_array([])
fig.colorbar(sm, cax=cax, label='Peak source power (mW)')

fig.savefig(OUTPUT_DIR / 'power_sweep_overlay.png', dpi=150)
plt.show()

In [ ]:
# ── Heatmap: power level × time, one per condition + shared IO curve ──────
_cond_info = [
    ('opt',  'Optimised',                'tomato'),
    ('rect', 'Energy-matched\nsquare pulse', 'steelblue'),
    ('pt',   f'Energy-matched\npulse train ({CMP_PT_FREQ_HZ:.0f} Hz)', 'seagreen'),
]

fig, axes = plt.subplots(1, 4, figsize=(22, 4),
                          gridspec_kw={'width_ratios': [3, 3, 3, 1.8]})

for ax, (ckey, ctitle, _) in zip(axes[:3], _cond_info):
    heatmap = np.stack([r[ckey]['mean_hz'] for r in sweep_results])
    im = ax.imshow(
        heatmap, aspect='auto', origin='lower',
        extent=[t_psth_ms[0], t_psth_ms[-1], power_levels[0], power_levels[-1]],
        cmap='inferno',
    )
    fig.colorbar(im, ax=ax, label='Firing rate (Hz)', shrink=0.9)
    ax.set_xlabel('Time from stim onset (ms)')
    ax.set_ylabel('Peak source power (mW)')
    ax.set_title(ctitle)

# Input–output curves — power on x, peak firing rate on y
ax = axes[3]
for ckey, ctitle, col in _cond_info:
    peak_arr = np.array([r[ckey]['peak_hz'] for r in sweep_results])
    ax.plot(power_levels, peak_arr, 'o-', color=col, lw=1.8,
            label=ctitle.replace('\n', ' '))
ax.set_xlabel('Peak source power (mW)')
ax.set_ylabel('Peak firing rate (Hz)')
ax.set_title('Input–output curves')
ax.legend(frameon=False, fontsize=7)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'power_sweep_heatmap.png', dpi=150)
plt.show()

In [ ]:
_save = {
    'source':               _source,
    'target_layer':         _layer,
    'opt_result_path':      str(OPT_RESULT_PATH),
    'power_levels_mw':      power_levels,
    'n_runs':               N_RUNS,
    'seed_base':            SEED_BASE,
    'bin_size_ms':          BIN_SIZE_MS,
    'stim_dur_ms':          STIM_DUR_MS,
    'orig_stim_dur_ms':     _orig_stim_dur,
    'pre_silence_ms':       PRE_SILENCE_MS,
    'post_silence_ms':      POST_SILENCE_MS,
    'native_peak_mw':       _peak_native,
    'native_energy_mwms':   _energy_native,
    'native_avg_power_mw':  _avg_power_native,
    'cmp_pt_freq_hz':       CMP_PT_FREQ_HZ,
    'cmp_pt_pulse_dur_ms':  CMP_PT_PULSE_DUR_MS,
    'pt_unit_energy_mwms':  _pt_unit_energy,
    't_psth_ms':            t_psth_ms,
    # sweep_results: list of dicts, one per power level.
    # Each dict has 'power_mw' and sub-dicts 'opt', 'rect', 'pt'
    # each with keys 'mean_hz', 'sem_hz', 'peak_hz'.
    'sweep_results':        sweep_results,
}
_save_path = OUTPUT_DIR / 'power_sweep_results.pkl'
with open(_save_path, 'wb') as f:
    pickle.dump(_save, f)
print(f'Saved → {_save_path}')

## Fine low-power sweep

Separate run over `FINE_MIN_POWER_MW` → `FINE_MAX_POWER_MW` with finer discretisation.
Kept separate from the wide sweep above so the colour axis is not compressed.

In [ ]:
fine_power_levels = np.linspace(FINE_MIN_POWER_MW, FINE_MAX_POWER_MW, FINE_N_LEVELS)
print(f'Fine sweep: {FINE_N_LEVELS} levels  '
      f'{FINE_MIN_POWER_MW:.3f} – {FINE_MAX_POWER_MW:.3f} mW')
print('  ' + ', '.join(f'{p:.3f}' for p in fine_power_levels) + ' mW')

fine_sweep_results = []

for idx, p_mw in enumerate(fine_power_levels):
    print(f'[{idx+1}/{len(fine_power_levels)}]  {p_mw:.4f} mW ...', flush=True)
    _level = {'power_mw': p_mw}

    for cname, maker in _makers:
        wf = maker(p_mw)
        _runs_hz = []
        for _i in range(N_RUNS):
            _r = model.run(wf, seed=SEED_BASE + _i,
                           vary_init_v=VARY_INIT_V,
                           vary_connectivity=VARY_CONNECTIVITY,
                           vary_weights=VARY_WEIGHTS)
            _runs_hz.append(_r['psth_exc'] / (BIN_SIZE_MS / 1000.0))
        _arr = np.stack(_runs_hz)
        _level[cname] = {
            'mean_hz': _arr.mean(0),
            'sem_hz':  _arr.std(0) / np.sqrt(N_RUNS),
            'peak_hz': float(_arr.mean(0).max()),
        }

    fine_sweep_results.append(_level)
    print(f'   peak — opt: {_level["opt"]["peak_hz"]:.1f} Hz  '
          f'rect: {_level["rect"]["peak_hz"]:.1f} Hz  '
          f'pt: {_level["pt"]["peak_hz"]:.1f} Hz')

print('Fine sweep complete.')

In [ ]:
# ── Fine sweep: waveform + PSTH overlay ──────────────────────────────────
cmap_f   = plt.cm.plasma
fp_norm  = plt.Normalize(fine_power_levels.min(), fine_power_levels.max())

_cond_info = [
    ('opt',  'Optimised',                _make_waveform),
    ('rect', 'Energy-matched square pulse', _make_rect),
    ('pt',   f'Energy-matched pulse train\n({CMP_PT_FREQ_HZ:.0f} Hz, {CMP_PT_PULSE_DUR_MS:.0f} ms pulses)', _make_pt),
]

fig, axes = plt.subplots(2, 3, figsize=(18, 6),
                          gridspec_kw={'height_ratios': [1, 2.5]},
                          sharex='col')

for col, (ckey, ctitle, maker) in enumerate(_cond_info):
    ax_wf   = axes[0, col]
    ax_psth = axes[1, col]

    for p_mw in fine_power_levels:
        color = cmap_f(fp_norm(p_mw))
        ax_wf.plot(_t_wf, maker(p_mw)(_t_wf), color=color, lw=1.2)
    ax_wf.set_xlim(0, STIM_DUR_MS)
    ax_wf.set_ylabel('Power (mW)')
    ax_wf.set_title(ctitle)
    ax_wf.tick_params(bottom=False, labelbottom=False)

    for res in fine_sweep_results:
        color = cmap_f(fp_norm(res['power_mw']))
        d = res[ckey]
        ax_psth.fill_between(t_psth_ms, d['mean_hz'] - d['sem_hz'], d['mean_hz'] + d['sem_hz'],
                             color=color, alpha=0.15)
        ax_psth.plot(t_psth_ms, d['mean_hz'], color=color, lw=1.5)
    ax_psth.set_xlim(0, STIM_DUR_MS)
    ax_psth.set_xlabel('Time from stim onset (ms)')

axes[1, 0].set_ylabel('Firing rate (Hz)')
fig.suptitle(f'{_source} / {_layer}  —  fine low-power sweep  ({N_RUNS} runs each)', y=1.01)

fig.tight_layout()
fig.subplots_adjust(right=0.87)
cax = fig.add_axes([0.89, 0.1, 0.015, 0.8])
sm = plt.cm.ScalarMappable(cmap=cmap_f, norm=fp_norm)
sm.set_array([])
fig.colorbar(sm, cax=cax, label='Peak source power (mW)')

fig.savefig(OUTPUT_DIR / 'fine_power_sweep_overlay.png', dpi=150)
plt.show()

# ── Fine sweep: heatmap + IO curve ───────────────────────────────────────
_cond_cols = [
    ('opt',  'Optimised',                'tomato'),
    ('rect', 'Energy-matched\nsquare pulse', 'steelblue'),
    ('pt',   f'Energy-matched\npulse train ({CMP_PT_FREQ_HZ:.0f} Hz)', 'seagreen'),
]

fig, axes = plt.subplots(1, 4, figsize=(22, 4),
                          gridspec_kw={'width_ratios': [3, 3, 3, 1.8]})

for ax, (ckey, ctitle, _) in zip(axes[:3], _cond_cols):
    heatmap = np.stack([r[ckey]['mean_hz'] for r in fine_sweep_results])
    im = ax.imshow(
        heatmap, aspect='auto', origin='lower',
        extent=[t_psth_ms[0], t_psth_ms[-1], fine_power_levels[0], fine_power_levels[-1]],
        cmap='inferno',
    )
    fig.colorbar(im, ax=ax, label='Firing rate (Hz)', shrink=0.9)
    ax.set_xlabel('Time from stim onset (ms)')
    ax.set_ylabel('Peak source power (mW)')
    ax.set_title(ctitle)

ax = axes[3]
for ckey, ctitle, col in _cond_cols:
    peak_arr = np.array([r[ckey]['peak_hz'] for r in fine_sweep_results])
    ax.plot(fine_power_levels, peak_arr, 'o-', color=col, lw=1.8,
            label=ctitle.replace('\n', ' '))
ax.set_xlabel('Peak source power (mW)')
ax.set_ylabel('Peak firing rate (Hz)')
ax.set_title('Input–output (fine range)')
ax.legend(frameon=False, fontsize=7)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'fine_power_sweep_heatmap.png', dpi=150)
plt.show()